# 02 — Download daily stock prices

**Goal:** daily prices for every ticker plus the market benchmark (SPY), from Jan 2024 to the end of April 2025. That covers the market-model estimation window (trading days −130 to −11) and the event windows.

| | |
|---|---|
| **Inputs** | Yahoo Finance via `yfinance` (free, no key) |
| **Outputs** | `data/raw/market_data/daily_prices.csv` in long format: one row per (date, ticker) with `close`, `adj_close`, `volume` |

**Why both close and adjusted close?** Returns use `adj_close` (it accounts for dividends and splits). Market cap uses the raw `close`, because shares outstanding from the 10-K are in raw, unadjusted shares.

*Known limitation (document in the README):* Yahoo data can be revised and has no delisted firms. CRSP would be the gold standard. This matters less here because every sample firm traded through April 2025.

In [ ]:
# --- Setup: find the repo root and load shared settings from code/config.py ---
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "run_all.py").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Can't find the repo root. Run this from inside the project folder.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "code"))

import config as cfg
print("Repo root:", ROOT)

In [ ]:
import pandas as pd
import yfinance as yf

cfg.MARKET_DIR.mkdir(parents=True, exist_ok=True)
symbols = cfg.TICKERS + [cfg.MARKET]

# auto_adjust=False so that yfinance returns BOTH "Close" and "Adj Close"
raw = yf.download(symbols, start=cfg.PRICE_START, end=cfg.PRICE_END,
                  auto_adjust=False, progress=False, group_by="column")
raw.tail(3)

## Reshape to long format (one row per date and ticker)

In [ ]:
pieces = []
for field, name in [("Close", "close"), ("Adj Close", "adj_close"), ("Volume", "volume")]:
    wide = raw[field].copy()
    wide.index.name = "date"
    pieces.append(wide.reset_index().melt(id_vars="date", var_name="ticker", value_name=name))

prices = pieces[0].merge(pieces[1], on=["date", "ticker"]).merge(pieces[2], on=["date", "ticker"])
prices = prices.dropna(subset=["adj_close"]).sort_values(["ticker", "date"]).reset_index(drop=True)
prices.to_csv(cfg.PRICES_CSV, index=False)
print(f"Saved {len(prices):,} rows to {cfg.PRICES_CSV.relative_to(cfg.ROOT)}")

# Provenance: record when and how the prices were pulled (this small file IS committed to git)
from datetime import datetime, timezone
cfg.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
pd.DataFrame([{
    "source": "Yahoo Finance via yfinance", "yfinance_version": getattr(yf, "__version__", "unknown"),
    "symbols": " ".join(symbols), "start": cfg.PRICE_START, "end_exclusive": cfg.PRICE_END,
    "rows": len(prices), "downloaded_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M"),
}]).to_csv(cfg.PROCESSED_DIR / "prices_download_log.csv", index=False)

## Checks

In [ ]:
coverage = prices.groupby("ticker")["date"].agg(["min", "max", "count"])
print(coverage)

missing = sorted(set(symbols) - set(prices["ticker"]))
if missing:
    # Log the problem instead of failing silently
    probs = pd.read_csv(cfg.DOWNLOAD_PROBLEMS_CSV)
    new = pd.DataFrame({"ticker": missing, "step": "02_prices", "problem": "no price data from yfinance"})
    pd.concat([probs, new]).to_csv(cfg.DOWNLOAD_PROBLEMS_CSV, index=False)
    print("WARNING: no prices for", missing)

assert cfg.MARKET in set(prices["ticker"]), "market benchmark missing: cannot compute abnormal returns"
assert pd.Timestamp(cfg.EVENT_DAY0) in set(pd.to_datetime(prices["date"])), "event day 0 missing from price data"
assert not prices.duplicated(["date", "ticker"]).any(), "duplicate (date, ticker) rows"